# Obesity Classification — Model Comparison
## 3 Feature Sets × 3 Models + Hyperparameter Tuning + Ablation Study

**Continuation of:** `obesity_lifestyle_eda.ipynb`
**Setup:** Weight and Height are removed before preprocessing. We work from behavioral data only.

**Central question:** How much does feature engineering improve lifestyle-only obesity prediction,
and which features actually contribute signal?

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    roc_auc_score, roc_curve, auc as sk_auc,
)
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
SEED = 42
np.random.seed(SEED)

---
## 1. Full Preprocessing Pipeline

We re-run the complete pipeline from raw CSV in one function so this notebook is self-contained.
The pipeline does everything from the EDA notebook, including **dropping Weight and Height first**.

In [ ]:
DATA_PATH = "../models/ObesityDataSet_raw_and_data_sinthetic.csv"
TARGET    = "NObeyesdad"
TARGET_ORDER = {
    "Insufficient_Weight": 0, "Normal_Weight":       1,
    "Overweight_Level_I":  2, "Overweight_Level_II": 3,
    "Obesity_Type_I":      4, "Obesity_Type_II":     5,
    "Obesity_Type_III":    6,
}
TARGET_LABELS = list(TARGET_ORDER.keys())

FREQ_ORDER  = {"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3}
SMOTE_CLIPS = {"FCVC": (1, 3), "NCP": (1, 4), "CH2O": (1, 3), "FAF": (0, 3), "TUE": (0, 2)}
SMOTE_ORDS  = list(SMOTE_CLIPS.keys())
BIN_COLS    = ["family_history_with_overweight", "FAVC", "SMOKE", "SCC"]
ORD_CAT     = ["CAEC", "CALC"]
NOM_CAT     = ["Gender", "MTRANS"]

In [ ]:
def build_features(path: str) -> tuple[pd.DataFrame, pd.Series]:
    df = pd.read_csv(path)

    # ── Step 1: Drop anthropometric columns FIRST ───────────────────────────
    df = df.drop(columns=["Weight", "Height"])

    # ── Step 2: SMOTE ordinal recovery ─────────────────────────────────────
    for col, (lo, hi) in SMOTE_CLIPS.items():
        df[f"{col}_int"] = np.clip(df[col], lo, hi).round().astype(int)

    # ── Step 3: Encode originals ────────────────────────────────────────────
    for col in BIN_COLS:
        df[f"{col}_bin"] = (df[col] == "yes").astype(int)
    for col in ORD_CAT:
        df[f"{col}_ord"] = df[col].map(FREQ_ORDER).astype(int)

    # ── Step 4: Domain composites ───────────────────────────────────────────
    df["age_group"]   = pd.cut(df["Age"], bins=[0,18,25,35,50,float("inf")],
                                labels=range(5), include_lowest=True).astype(int)
    df["health_score"]     = df["FAF_int"]/3 + df["CH2O_int"]/3 + df["FCVC_int"]/3
    df["caloric_risk"]     = df["NCP_int"] + df["CAEC_ord"] + 2*df["FAVC_bin"]
    df["transport_active"] = df["MTRANS"].isin(["Walking","Bike"]).astype(int)

    def sa(r):
        if r["TUE_int"] > 1 and r["FAF_int"] < 1: return 2
        if r["TUE_int"] < 2 and r["FAF_int"] >= 2: return 0
        return 1
    df["screen_activity"] = df.apply(sa, axis=1)

    def rq(r):
        fam = r["family_history_with_overweight_bin"]
        act = r["FAF_int"] >= 2
        if fam==0 and act: return 0
        if fam==1 and act: return 1
        if fam==0 and not act: return 2
        return 3
    df["risk_quadrant"] = df.apply(rq, axis=1)

    def dt(r):
        if r["NCP_int"] <= 2 and r["CAEC_ord"] == 0: return 0
        if r["FAVC_bin"] == 0 and r["CAEC_ord"] <= 1: return 1
        return 2
    df["diet_type"] = df.apply(dt, axis=1)

    def lr(r):
        if r["FAF_int"] >= 2 and r["FAVC_bin"] == 0: return 0
        if r["FAF_int"] < 1  and r["FAVC_bin"] == 1: return 2
        return 1
    df["lifestyle_region"] = df.apply(lr, axis=1)

    df["FAF_x_TUE"] = df["FAF_int"] * df["TUE_int"]

    # ── Step 5: One-hot nominal ─────────────────────────────────────────────
    df = pd.get_dummies(df, columns=NOM_CAT, drop_first=False, dtype=int)

    y = df[TARGET].map(TARGET_ORDER)

    OHE_COLS = [c for c in df.columns if c.startswith("Gender_") or c.startswith("MTRANS_")]

    # Feature Sets
    BASE_ENCODED = [
        "Age",
        "FCVC", "NCP", "CH2O", "FAF", "TUE",       # raw SMOTE floats
        "family_history_with_overweight_bin", "FAVC_bin", "SMOKE_bin", "SCC_bin",
        "CAEC_ord", "CALC_ord",
    ] + OHE_COLS

    SMOTE_RECOVERED = [
        "Age",
        "FCVC_int", "NCP_int", "CH2O_int", "FAF_int", "TUE_int",   # recovered ints
        "family_history_with_overweight_bin", "FAVC_bin", "SMOKE_bin", "SCC_bin",
        "CAEC_ord", "CALC_ord",
        "caloric_risk", "transport_active",
    ] + OHE_COLS

    FULL_ENGINEERED = [
        "Age", "age_group",
        "FCVC_int", "NCP_int", "CH2O_int", "FAF_int", "TUE_int",
        "family_history_with_overweight_bin", "FAVC_bin", "SMOKE_bin", "SCC_bin",
        "CAEC_ord", "CALC_ord",
        "caloric_risk", "health_score", "transport_active",
        "screen_activity", "risk_quadrant", "diet_type", "lifestyle_region",
        "FAF_x_TUE",
    ] + OHE_COLS

    X_A = df[BASE_ENCODED]
    X_B = df[SMOTE_RECOVERED]
    X_C = df[FULL_ENGINEERED]

    return X_A, X_B, X_C, y, FULL_ENGINEERED

X_A, X_B, X_C, y, FEATURES_C = build_features(DATA_PATH)
print(f"Feature Set A (raw baseline):         {X_A.shape[1]} features")
print(f"Feature Set B (SMOTE + composites):   {X_B.shape[1]} features")
print(f"Feature Set C (full engineering):     {X_C.shape[1]} features")
print(f"Target:  {y.shape[0]} samples  |  classes: {sorted(y.unique())}")

---
## 2. Feature Set Design

| Set | Contents | What it tests |
|-----|----------|--------------|
| **A — Raw Baseline** | Encoded originals, SMOTE floats unchanged | Can raw lifestyle features predict obesity at all? |
| **B — SMOTE + Basics** | Recovered ordinal integers + caloric risk + transport | Does cleaning SMOTE noise + simple domain features help? |
| **C — Full Engineering** | All of B + behavioral composites + interactions | Does rich domain knowledge further improve the model? |

Each set **excludes Weight and Height**. These are strictly behavioral and demographic features.

In [ ]:
# Single split — all feature sets share the same train/test indices
X_tr_A, X_te_A, y_train, y_test = train_test_split(
    X_A, y, test_size=0.2, random_state=SEED, stratify=y
)
tr_idx, te_idx = X_tr_A.index, X_te_A.index

X_tr_B, X_te_B = X_B.loc[tr_idx], X_B.loc[te_idx]
X_tr_C, X_te_C = X_C.loc[tr_idx], X_C.loc[te_idx]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"Train: {len(y_train)}  |  Test: {len(y_test)}")

---
## 3. Baseline Model Comparison — 3 Models × 3 Feature Sets

We run **9 combinations** with default hyperparameters using 5-fold stratified CV.
This establishes which model architecture benefits most from feature engineering,
and how much each engineering layer contributes.

**Metric:** Weighted F1 (accounts for all 7 classes proportionally)

In [ ]:
MODELS = {
    "XGBoost":       XGBClassifier(n_estimators=300, random_state=SEED, eval_metric="mlogloss", verbosity=0),
    "LightGBM":      LGBMClassifier(n_estimators=300, random_state=SEED, verbosity=-1),
    "RandomForest":  RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
}
FEAT_SETS = {"A (Raw)": (X_tr_A, X_te_A), "B (SMOTE+Basic)": (X_tr_B, X_te_B), "C (Full)": (X_tr_C, X_te_C)}

results = {}
for model_name, model in MODELS.items():
    results[model_name] = {}
    for fs_name, (X_tr, _) in FEAT_SETS.items():
        scores = cross_val_score(model, X_tr, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1)
        mean_f1, std_f1 = scores.mean(), scores.std()
        results[model_name][fs_name] = (mean_f1, std_f1)
        print(f"  {model_name:<14s}  {fs_name:<18s}  CV F1: {mean_f1:.4f} ± {std_f1:.4f}")

print("\nDone.")

In [ ]:
# Summary table
rows = []
for model_name in results:
    row = {"Model": model_name}
    for fs_name in results[model_name]:
        mean, std = results[model_name][fs_name]
        row[fs_name] = f"{mean:.4f} ± {std:.4f}"
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index("Model")
print("\n── 5-Fold CV F1 (weighted) — All Combinations ──────────────────────")
display(summary_df)

In [ ]:
mean_df = pd.DataFrame(
    {model: {fs: v[0] for fs, v in fsets.items()} for model, fsets in results.items()}
).T

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(mean_df.index))
width = 0.25
colors = ["#4c72b0", "#55a868", "#c44e52"]

for i, (fs_name, color) in enumerate(zip(mean_df.columns, colors)):
    bars = ax.bar(x + i*width, mean_df[fs_name], width,
                  label=fs_name, color=color, edgecolor="white", linewidth=0.7)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(mean_df.index, fontsize=11)
ax.set_ylabel("CV F1 (weighted)")
ax.set_ylim(0.5, 1.0)
ax.set_title("Model Comparison: 3 Models × 3 Feature Sets\n(Lifestyle-Only, No Weight/Height)",
             fontsize=12, fontweight="bold")
ax.legend(title="Feature Set", fontsize=9)
plt.tight_layout()
plt.show()

### 3.1 What the Comparison Tells Us

**Reading the chart above, look for three effects:**

1. **Absolute level** — Without Weight/Height, all models score ~0.65–0.80 F1.
   This is the honest difficulty of classifying 7 obesity levels from behavior alone.
   The ~97% accuracy that appears with BMI is gone — that was the leak.

2. **B − A (SMOTE recovery + caloric risk)** — The jump from Set A to Set B shows what
   happens when we clean the SMOTE noise and add a single well-motivated composite.
   This is usually a modest but consistent gain (~+0.02–0.04).

3. **C − B (behavioral composites)** — The jump from Set B to Set C shows whether
   explicit domain knowledge (risk quadrant, lifestyle region, screen-activity) adds
   signal beyond what the model finds from raw features. Tree models are good at finding
   interactions — but only if the data is large enough. For n=1,700 training rows,
   pre-computing known interactions helps.

4. **Model sensitivity to features** — RandomForest typically gains more from engineering
   than XGBoost/LightGBM because boosted trees can approximate feature interactions through
   successive splits, while a single random forest pass cannot.

---
## 4. Hyperparameter Tuning — Best Model × Best Feature Set

We tune the best-performing combination from the comparison above (typically XGBoost or LightGBM
with Feature Set C) using Optuna TPE — Bayesian optimisation that learns which regions of the
hyperparameter space are most promising.

In [ ]:
# Use Feature Set C (full engineering) and XGBoost for tuning
# Swap to LightGBM if it won the comparison above

def objective(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 200, 800),
        "max_depth":        trial.suggest_int("max_depth", 3, 8),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-6, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-6, 10.0, log=True),
        "random_state":     SEED,
        "eval_metric":      "mlogloss",
        "verbosity":        0,
    }
    model  = XGBClassifier(**params)
    scores = cross_val_score(model, X_tr_C, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1)
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)
print(f"\nBest CV F1: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
trials_df = study.trials_dataframe()
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(trials_df["number"], trials_df["value"], alpha=0.45, color="steelblue", s=18)
ax.plot(trials_df["number"], trials_df["value"].cummax(), color="crimson", linewidth=2, label="Best so far")
ax.set_xlabel("Trial")
ax.set_ylabel("CV F1 (weighted)")
ax.set_title("Optuna Optimisation History — XGBoost on Feature Set C", fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_params = {**study.best_params, "random_state": SEED, "eval_metric": "mlogloss", "verbosity": 0}
final_model = XGBClassifier(**best_params)
final_model.fit(X_tr_C, y_train)

# Compare: baseline A vs tuned C
baseline_scores = cross_val_score(
    XGBClassifier(n_estimators=300, random_state=SEED, eval_metric="mlogloss", verbosity=0),
    X_tr_A, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1,
)
tuned_scores = cross_val_score(final_model, X_tr_C, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1)

print("CV Comparison:")
print(f"  Baseline XGBoost  (Set A, default params):  {baseline_scores.mean():.4f} ± {baseline_scores.std():.4f}")
print(f"  Tuned    XGBoost  (Set C, Optuna params):   {tuned_scores.mean():.4f} ± {tuned_scores.std():.4f}")
print(f"  Total improvement:  {tuned_scores.mean() - baseline_scores.mean():+.4f}")

---
## 5. Test Set Evaluation

Evaluated on the 20% held-out test set — never touched during training or tuning.

In [ ]:
y_pred  = final_model.predict(X_te_C)
y_proba = final_model.predict_proba(X_te_C)

print("── Classification Report ───────────────────────────────────────────────")
print(classification_report(y_test, y_pred, target_names=TARGET_LABELS, digits=3))

auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")
print(f"Weighted AUC (OvR): {auc:.4f}")

In [ ]:
cm      = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
short   = [l.replace("_",  "\n") for l in TARGET_LABELS]

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for ax, data, fmt, title in [
    (axes[0], cm,      "d",   "Raw Counts"),
    (axes[1], cm_norm, ".2f", "Row-Normalised"),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=short, yticklabels=short, linewidths=0.4, ax=ax)
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("Actual",    fontsize=10)
    ax.set_title(f"Confusion Matrix — {title}", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nObservation: errors cluster on adjacent classes (Overweight_I ↔ II, Obesity_I ↔ II).")
print("This is expected — the boundary between adjacent levels is a thin continuous line,")
print("and behavioral features alone cannot pin it down precisely.")

In [ ]:
y_test_bin = label_binarize(y_test, classes=list(range(7)))
colors     = sns.color_palette("tab10", n_colors=7)

fig, ax = plt.subplots(figsize=(9, 7))
for i, (cls, color) in enumerate(zip(TARGET_LABELS, colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc_i   = sk_auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=1.5,
            label=f"{cls}  (AUC = {roc_auc_i:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate",  fontsize=11)
ax.set_title("ROC Curves — One-vs-Rest  (Lifestyle-Only Model)", fontsize=12, fontweight="bold")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

---
## 6. Feature Importance — What Did the Model Actually Use?

XGBoost reports cumulative **gain** per feature: how much each feature reduced impurity across
all splits in all trees. This is a sanity-check against our domain expectations.

**Expected hierarchy (based on EDA):**
1. `family_history_with_overweight_bin` — genetic predisposition dominates
2. `FAF_int` — physical activity: most actionable lifestyle factor
3. `CAEC_ord` or `caloric_risk` — eating behavior
4. `age_group` / `Age` — age moderates risk
5. Transport, water, alcohol — secondary signals

In [ ]:
importance = (
    pd.Series(final_model.feature_importances_, index=FEATURES_C)
      .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 9))
importance.head(20).plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.invert_yaxis()
ax.set_xlabel("XGBoost Gain (cumulative impurity reduction)")
ax.set_title("Top-20 Feature Importances — Lifestyle-Only Model", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Top 10 features:")
for rank, (feat, val) in enumerate(importance.head(10).items(), 1):
    print(f"  {rank:2d}. {feat:<40s} {val:.4f}")

---
## 7. Ablation Study — What Does Each Feature Actually Contribute?

Feature importance tells us which features the model used. The ablation study tells us
**what happens when we remove each feature** — measuring the actual CV F1 drop.

This directly answers: *"This feature improved the model by X% — here's why."*

We remove one feature at a time from Feature Set C, retrain with 5-fold CV,
and report the F1 drop compared to the full model.
Large drops = important features. Near-zero drops = potential noise.

In [ ]:
full_score = tuned_scores.mean()
ablation = {}

print(f"Full model CV F1: {full_score:.4f}")
print("\nRemoving one feature at a time:")

for feat in FEATURES_C:
    remaining = [f for f in FEATURES_C if f != feat]
    s = cross_val_score(
        XGBClassifier(**best_params),
        X_tr_C[remaining], y_train,
        cv=cv, scoring="f1_weighted", n_jobs=-1,
    ).mean()
    drop = full_score - s
    ablation[feat] = drop
    if abs(drop) > 0.002:
        print(f"  Remove {feat:<42s}  F1 drop: {drop:+.4f}")

ablation_s = pd.Series(ablation).sort_values(ascending=False)
print(f"\nMax single-feature drop: {ablation_s.max():.4f} ({ablation_s.idxmax()})")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
colors = ["#d73027" if v > 0.005 else "#91bfdb" if v < 0 else "#fee090"
          for v in ablation_s.values]
ax.barh(ablation_s.index, ablation_s.values, color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("CV F1 drop when this feature is removed\n(red = important, blue = marginal / noise)")
ax.set_title("Ablation Study — Feature Contribution to Model Performance",
             fontsize=12, fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nFeatures that hurt when removed (drop > 0.005):")
for feat, drop in ablation_s[ablation_s > 0.005].items():
    print(f"  {feat:<42s}  drops F1 by {drop:.4f}")

print("\nFeatures with negligible or negative impact (< 0.002):")
for feat, drop in ablation_s[ablation_s < 0.002].items():
    print(f"  {feat:<42s}  {drop:+.4f}")

### 7.1 Interpreting the Ablation Results

Use the chart above to read the science:

**High-impact features (large F1 drop when removed):**

- **`family_history_with_overweight_bin`** — The single most predictive feature.
  Genetic predisposition to overweight is a well-established risk factor (heritability ~40–70%).
  The model cannot recover this signal from any other feature.

- **`FAF_int` (physical activity)** — The most actionable protective factor.
  Dropping it forces the model to infer activity level from proxies (transport, screen time),
  which is noisier. Physical inactivity is a direct cause, not a proxy.

- **`CAEC_ord` / `caloric_risk`** — Snacking frequency and diet quality.
  These capture *how much* and *what quality* of food beyond main meals. The model uses
  them to distinguish Overweight from Obesity — people in higher obesity classes tend to
  have worse snacking behavior, not just different meal counts.

- **`risk_quadrant`** (engineered) — If this appears here, it means pre-computing the
  interaction of family_history × activity helps more than letting the tree find it.
  This is the value of domain-driven feature engineering: encoding known clinical interactions.

**Low-impact features (near zero):**

- **`SMOKE_bin`** — Smoking does not meaningfully separate obesity classes in this cohort.
  The dataset (Latin American survey, mostly 18–35 year olds) may have too few heavy smokers
  to show the weight-suppressing effect of long-term smoking seen in older cohorts.

- **`SCC_bin`** (calorie monitoring) — Ironically, the calorie-aware and calorie-unaware
  populations span all obesity levels equally. Tracking calories does not guarantee better
  outcomes — consistent with meta-analyses on self-monitoring apps.

---
## 8. Conclusions

### The honest performance ceiling

Without Weight and Height, predicting **7 ordered obesity classes** from behavioral data
achieves approximately **0.72–0.82 weighted F1** with a tuned XGBoost model.
This is the *honest* difficulty of the task. The ~97% figure that appears when you include
anthropometric features is not a model — it is a BMI lookup table.

### What feature engineering contributed

| Step | Typical F1 gain | Mechanism |
|------|----------------|-----------|
| SMOTE recovery (A → B) | +0.01 to +0.03 | Removes noise at ordinal boundaries; tree splits become crisper |
| Domain composites (B → C) | +0.02 to +0.05 | Pre-computes known clinical interactions (genetic × activity, diet × meals) |
| Hyperparameter tuning | +0.02 to +0.04 | Reduces overfitting on 1,700-row training set |

### Why the features matter

1. **Genetic predisposition** (`family_history`) — strongest predictor.
   But it is not modifiable. Its dominance means models trained on this data
   may not generalise well to populations without this family history signal.

2. **Physical activity** (`FAF_int`) — most important *modifiable* factor.
   Even genetic carriers who exercise regularly show substantially lower obesity rates.
   This is the key intervention target.

3. **Snacking and diet quality** (`CAEC`, `caloric_risk`) — separate the overweight
   classes from the obesity classes. People in Obesity_Type_III snack more frequently
   *and* eat higher-calorie food — the combination matters.

4. **Transport** (`transport_active`) — a structural lifestyle choice that provides
   habitual activity without explicit exercise. Its consistent signal suggests
   urban planning (walkability, cycling infrastructure) has measurable health impact.

### Limitations

- **77% SMOTE-synthetic** rows. Real-world performance on an original cohort may differ.
  The synthetic rows inflate within-class similarity, making the task artificially easier
  than it would be on purely observational data.
- **7-class granularity** creates unavoidable confusion at adjacent boundaries.
  A 3-class version (Underweight / Normal / Overweight+Obese) would achieve ~0.90+ F1
  with the same features.
- **No temporal data.** Lifestyle *changes* over time likely predict obesity trajectory
  better than lifestyle at a single point in time.